# Lab 3 — Evaluate base vs fine-tuned OpenVLA

A VLA outputs numbers, not text, so the serverless lab's LLM-as-judge and text scorers don't apply. We
score **closed-form** instead: for each validation frame, greedy-decode the 7 action tokens and compare
them with the label:

* `l1_mean`: mean absolute error over all 7 axes (normalized units, [-1, 1])
* `l1_x`, `l1_y`: the two axes that carry signal in this pseudo-labelled data
* `bin_accuracy`: fraction of axes whose prediction falls in the same 1/256 bin as the label
* `mean_action_baseline`: always predict the mean training action. A model that doesn't beat it has
  learned nothing beyond the label prior.

The comparison runs as a single-GPU SageMaker job (`scripts/evaluate_openvla.py`), so the notebook
kernel needs no GPU. The job evaluates the base model first, then the merged model.

In [ ]:
import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

import config

sess = Session()
try:
    role = get_execution_role()
except ValueError:
    role = boto3.client("iam").get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

region = sess.boto_region_name
bucket_name = sess.default_bucket()
s3_root = config.s3_root(bucket_name, sess.default_bucket_prefix)
dataset_s3_uri = f"{s3_root}/datasets/{config.DATASET_PREFIX}/"
s3 = boto3.client("s3", region_name=region)
sm_client = boto3.client("sagemaker", region_name=region)

print(f"role:     {role}")
print(f"region:   {region}")
print(f"run mode: {config.RUN_MODE} -> {config.TRAIN['instance_type']}")
print(f"s3 root:  {s3_root}")

In [ ]:
def latest_package(status=None):
    kwargs = {"ModelPackageGroupName": config.MODEL_PACKAGE_GROUP, "SortBy": "CreationTime", "SortOrder": "Descending"}
    if status:
        kwargs["ModelApprovalStatus"] = status
    pkgs = sm_client.list_model_packages(**kwargs)["ModelPackageSummaryList"]
    if not pkgs:
        raise RuntimeError(f"no {status or ''} model package in {config.MODEL_PACKAGE_GROUP}; run the previous notebook")
    return sm_client.describe_model_package(ModelPackageName=pkgs[0]["ModelPackageArn"])

In [ ]:
from config import resolve_mlflow_app

mlflow_app_arn = resolve_mlflow_app(sm_client, session=sess)
package = latest_package()
merged_s3_uri = package["CustomerMetadataProperties"]["merged_s3_uri"]
print(f"package: {package['ModelPackageArn']} ({package['ModelApprovalStatus']})\nmodel:   {merged_s3_uri}")

In [ ]:
import time

def wait_for_job(job_name, poll=60):
    """Prints status transitions and the latest [step ...] log line instead of streaming every log line
    (the full stream can exceed Jupyter's output rate limit)."""
    logs = boto3.client("logs", region_name=region)
    print(f"https://{region}.console.aws.amazon.com/sagemaker/home?region={region}#/jobs/{job_name}")
    last = None
    while True:
        d = sm_client.describe_training_job(TrainingJobName=job_name)
        step = ""
        try:
            events = logs.filter_log_events(logGroupName="/aws/sagemaker/TrainingJobs", logStreamNamePrefix=job_name,
                                            filterPattern='"[step"', startTime=int((time.time() - 900) * 1000))["events"]
            step = events[-1]["message"].strip()[:160] if events else ""
        except logs.exceptions.ResourceNotFoundException:
            pass
        line = f"{d['TrainingJobStatus']}/{d['SecondaryStatus']} {step}"
        if line != last:
            print(time.strftime("%H:%M:%S"), line, flush=True)
            last = line
        if d["TrainingJobStatus"] in ("Completed", "Failed", "Stopped"):
            if d["TrainingJobStatus"] != "Completed":
                raise RuntimeError(d.get("FailureReason", d["TrainingJobStatus"]))
            return d
        time.sleep(poll)

In [ ]:
from sagemaker.core.training.configs import Compute, InputData, OutputDataConfig, SourceCode, StoppingCondition
from sagemaker.train import ModelTrainer

eval_name = f"{config.PROJECT}-eval-{time.strftime('%Y%m%d-%H%M%S')}"
evaluator = ModelTrainer(
    training_image=config.TRAINING_IMAGE.format(region=region),
    source_code=SourceCode(source_dir="scripts", requirements="requirements.txt", entry_script="run_evaluation.sh"),
    compute=Compute(instance_type=config.EVAL_INSTANCE_TYPE, instance_count=1, volume_size_in_gb=100,
                    enable_managed_spot_training=config.USE_SPOT),
    stopping_condition=StoppingCondition(max_runtime_in_seconds=2 * 3600,
                                         max_wait_time_in_seconds=config.SPOT_MAX_WAIT_H * 3600 if config.USE_SPOT else None),
    output_data_config=OutputDataConfig(s3_output_path=f"{s3_root}/eval-output", compression_type="NONE"),
    environment={"MLFLOW_TRACKING_URI": mlflow_app_arn, "MLFLOW_EXPERIMENT_NAME": config.MLFLOW_EXPERIMENT},
    base_job_name=eval_name[:48],
    role=role,
    sagemaker_session=sess,
)
evaluator.train(input_data_config=[InputData(channel_name="data", data_source=dataset_s3_uri),
                                   InputData(channel_name="model", data_source=merged_s3_uri)], wait=False)
eval_job = wait_for_job(evaluator._latest_training_job.training_job_name)

In [ ]:
import json

import pandas as pd

results_uri = eval_job["ModelArtifacts"]["S3ModelArtifacts"].rstrip("/") + "/eval_results.json"
bucket, key = results_uri[5:].split("/", 1)
results = json.loads(s3.get_object(Bucket=bucket, Key=key)["Body"].read())

cols = ["mean_action_baseline", "base", "finetuned"]
table = pd.DataFrame({c: results[c] for c in cols})
table["finetuned vs base"] = (table["finetuned"] - table["base"]) / table["base"].abs()
print(f"{results['num_samples']} validation samples ({results_uri})")
table.style.format("{:.3f}").format("{:+.0%}", subset=["finetuned vs base"])

## Approve the package

The gate compares `l1_xy`, the L1 on the two axes that carry signal. The 7-axis `l1_mean` is dominated by
the 5 constant axes, where the mean-action baseline is exact by construction.

* **smoke:** approve if the fine-tuned model beats the base model. This proves training moved the model in the
  right direction, so notebook 4 can deploy it.
* **full:** approve only if it also beats the mean-action baseline, i.e. it learned more than the label prior.

In [ ]:
l1_xy = {k: (results[k]["l1_x"] + results[k]["l1_y"]) / 2 for k in ("finetuned", "base", "mean_action_baseline")}
ft, base, prior = l1_xy["finetuned"], l1_xy["base"], l1_xy["mean_action_baseline"]
approve = ft < base if package["CustomerMetadataProperties"].get("run_mode") == "smoke" else ft < min(base, prior)
summary = f"l1_xy finetuned={ft:.3f} base={base:.3f} mean-action={prior:.3f}"
sm_client.update_model_package(
    ModelPackageArn=package["ModelPackageArn"],
    ModelApprovalStatus="Approved" if approve else "Rejected",
    ApprovalDescription=summary,
)
print("Approved" if approve else "Rejected", f"({summary}, run_mode={package['CustomerMetadataProperties'].get('run_mode')})")

Continue with **4-deployment.ipynb**.